In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# 1. Loading the cleaned dataset
df = pd.read_csv('data/processed/cleaned_diabetic_data.csv')

# Separating target from features
y = df['target']
X = df.drop(columns=['target'])

# 2. Defining column types
cat_cols = [
    'race', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 
    'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 
    'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 
    'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 
    'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

# The remaining columns treated as numerical/ordinal
num_cols = [col for col in X.columns if col not in cat_cols]


# ==========================================
# PREP TRACK 1: XGBoost (Native Categories)
# ==========================================
X_xgb = X.copy()
# XGBoost just needs the categoricals cast to the 'category' dtype
existing_cats = [col for col in cat_cols if col in X_xgb.columns]
X_xgb[existing_cats] = X_xgb[existing_cats].astype('category')

X_lr = X.copy()
# 1. One-Hot Encoding categoricals (drop_first=True prevents multicollinearity)
X_lr = pd.get_dummies(X_lr, columns=existing_cats, drop_first=True)

# XGBoost Split
X_train_xgb, X_test_xgb, y_train, y_test = train_test_split(
    X_xgb, y, test_size=0.2, random_state=42, stratify=y
)

# Logistic Regression Split
X_train_lr, X_test_lr, _, _ = train_test_split(
    X_lr, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Scaling the numerical columns for Logistic Regression for data leakage
scaler = StandardScaler()
X_train_lr[num_cols] = scaler.fit_transform(X_train_lr[num_cols])
X_test_lr[num_cols] = scaler.transform(X_test_lr[num_cols])

print(f"XGBoost Training Features: {X_train_xgb.shape[1]}")
print(f"Logistic Regression Training Features: {X_train_lr.shape[1]}")

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/fi/Documents/Projects/DTSC 2/hospital-readmission-project/venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <5A4ADBF3-2060-3ECB-AB2B-F8A248DDA0A7> /Users/fi/Documents/Projects/DTSC 2/hospital-readmission-project/venv/lib/python3.10/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/Syste